# Robustness Analysis

This notebook tests how sensitive the portfolio results are to key modeling choices.

The robustness analysis varies:

- estimation window
- rebalancing frequency
- volatility target
- maximum leverage

Estimation window and rebalancing frequency are evaluated on raw portfolios without volatility targeting.

Volatility-target and leverage sensitivity are evaluated around the base-case implementation of a 10% volatility target and a 2x leverage cap.

In [1]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd

import metrics
import backtesting

importlib.reload(metrics)
importlib.reload(backtesting)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

## Data and strategies

The same ETF universe, risk-free rate and portfolio strategies used in the main backtest are reused here.

Each robustness test changes one modeling assumption while keeping the others fixed.

In [2]:
prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

In [3]:
def run_robustness_returns(window=WINDOW, step=STEP, vol_target=VOL_TARGET,
                           max_leverage=MAX_LEVERAGE,
                           transaction_cost=TRANSACTION_COST,
                           funding_spread=FUNDING_SPREAD):

    portfolio_returns = {}

    for name, (strategy, strategy_groups) in strategies.items():
        port_ret, _, _, _, _ = backtesting.backtest(
            returns,
            window=window,
            step=step,
            allocation_func=strategy,
            groups=strategy_groups,
            vol_target=vol_target,
            max_leverage=max_leverage,
            risk_free_returns=risk_free_returns,
            transaction_cost=transaction_cost,
            funding_spread=funding_spread
        )

        portfolio_returns[name] = port_ret

    return pd.DataFrame(portfolio_returns)


def performance_table(port_returns):
    results = []

    for strategy in port_returns.columns:
        port_ret = port_returns[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    return pd.DataFrame(results).set_index("Strategy")

## Estimation window

The estimation window controls how much historical data is used to estimate portfolio inputs.

The comparison uses **2-year, 3-year and 5-year** rolling windows on **raw portfolios without volatility targeting**.

Because different window lengths produce different out-of-sample start dates, all results are aligned to a common evaluation period before computing performance metrics.

In [4]:
windows = {
    "2Y": 2 * 252,
    "3Y": 3 * 252,
    "5Y": 5 * 252
}

window_returns = {}

for label, window in windows.items():
    window_returns[label] = run_robustness_returns(window=window, vol_target=None)

common_index = (
    window_returns["2Y"].index
    .intersection(window_returns["3Y"].index)
    .intersection(window_returns["5Y"].index)
)

for label, port_returns in window_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Estimation Window — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.01%,4.33%,0.583,-12.14%
Max Geo,10.69%,15.83%,0.627,-32.42%
Max Sharpe,5.60%,10.31%,0.435,-26.50%
Carver,3.19%,3.11%,0.541,-10.36%
Hierarchical Inv Vol,3.37%,3.36%,0.557,-10.65%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.15%,4.27%,0.622,-11.70%
Max Geo,13.70%,15.45%,0.811,-27.77%
Max Sharpe,7.19%,9.52%,0.619,-24.99%
Carver,3.26%,3.08%,0.568,-10.07%
Hierarchical Inv Vol,3.45%,3.32%,0.585,-10.25%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,8.47%,12.11%,0.608,-23.74%
Inverse Vol,4.10%,4.17%,0.624,-11.80%
Max Geo,12.04%,15.93%,0.699,-30.41%
Max Sharpe,6.72%,9.02%,0.600,-26.02%
Carver,3.18%,3.01%,0.555,-10.16%
Hierarchical Inv Vol,3.36%,3.24%,0.571,-10.37%
SPY,14.20%,17.06%,0.776,-33.72%
60/40,9.68%,9.77%,0.841,-21.18%
Tech,19.12%,20.65%,0.878,-35.12%


## Rebalancing frequency

The rebalancing interval is varied between:

- weekly
- monthly
- quarterly

This test is performed on **raw portfolios without volatility targeting** to isolate the effect of rebalancing frequency.

In [5]:
steps = {
    "Weekly": 5,
    "Monthly": 21,
    "Quarterly": 63
}

step_returns = {}

for label, step in steps.items():
    step_returns[label] = run_robustness_returns(step=step, vol_target=None)

common_index = (
    step_returns["Weekly"].index
    .intersection(step_returns["Monthly"].index)
    .intersection(step_returns["Quarterly"].index)
)

for label, port_returns in step_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Rebalancing Frequency — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.40%,14.87%,0.459,-41.49%
Inverse Vol,3.78%,5.05%,0.478,-15.70%
Max Geo,10.86%,15.48%,0.652,-37.25%
Max Sharpe,5.80%,8.97%,0.516,-26.09%
Carver,3.18%,3.42%,0.518,-10.14%
Hierarchical Inv Vol,3.34%,3.74%,0.517,-10.32%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.31%,11.34%,0.635,-31.47%
Tech,16.43%,22.41%,0.728,-49.37%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.31%,14.60%,0.459,-41.61%
Inverse Vol,3.77%,4.94%,0.486,-15.47%
Max Geo,11.28%,16.22%,0.653,-39.34%
Max Sharpe,6.56%,9.06%,0.590,-24.99%
Carver,3.17%,3.38%,0.521,-10.07%
Hierarchical Inv Vol,3.33%,3.69%,0.521,-10.25%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.28%,11.11%,0.643,-31.48%
Tech,16.43%,22.41%,0.728,-49.37%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,7.27%,14.32%,0.462,-41.42%
Inverse Vol,3.75%,4.89%,0.486,-15.42%
Max Geo,11.13%,16.59%,0.634,-39.69%
Max Sharpe,6.34%,9.38%,0.551,-22.69%
Carver,3.14%,3.35%,0.516,-10.11%
Hierarchical Inv Vol,3.29%,3.66%,0.516,-10.29%
SPY,11.40%,19.74%,0.574,-51.48%
60/40,8.32%,10.93%,0.656,-31.20%
Tech,16.43%,22.41%,0.728,-49.37%


## Volatility target

The annual volatility target is varied between **5%, 10% and 15%**.

The leverage cap remains fixed at 2x, so some low-volatility strategies may be unable to reach the higher targets.

In [6]:
vol_targets = {
    "5%": 0.05,
    "10%": 0.10,
    "15%": 0.15
}

vol_target_returns = {}

for label, vol_target in vol_targets.items():
    vol_target_returns[label] = run_robustness_returns(vol_target=vol_target)

common_index = (
    vol_target_returns["5%"].index
    .intersection(vol_target_returns["10%"].index)
    .intersection(vol_target_returns["15%"].index)
)

for label, port_returns in vol_target_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Volatility Target — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,3.43%,5.35%,0.392,-16.06%
Inverse Vol,3.46%,5.47%,0.390,-15.63%
Max Geo,4.77%,5.24%,0.644,-12.34%
Max Sharpe,4.63%,4.99%,0.648,-14.51%
Carver,3.06%,5.01%,0.343,-17.89%
Hierarchical Inv Vol,3.21%,5.20%,0.361,-16.83%
SPY,4.10%,5.45%,0.503,-16.28%
60/40,4.56%,5.47%,0.583,-16.26%
Tech,5.00%,5.28%,0.682,-14.02%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.23%,10.86%,0.393,-30.50%
Inverse Vol,4.06%,9.01%,0.329,-29.73%
Max Geo,8.03%,10.57%,0.650,-24.03%
Max Sharpe,6.72%,8.40%,0.648,-22.58%
Carver,3.76%,6.73%,0.371,-21.25%
Hierarchical Inv Vol,4.00%,7.30%,0.379,-21.59%
SPY,6.63%,11.06%,0.507,-30.91%
60/40,7.41%,11.11%,0.571,-31.05%
Tech,8.47%,10.64%,0.683,-26.99%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,6.63%,16.57%,0.385,-42.95%
Inverse Vol,4.90%,9.99%,0.387,-29.90%
Max Geo,11.14%,16.01%,0.651,-34.54%
Max Sharpe,8.06%,11.29%,0.617,-28.55%
Carver,3.79%,6.77%,0.374,-21.25%
Hierarchical Inv Vol,4.09%,7.40%,0.386,-21.59%
SPY,8.94%,16.88%,0.507,-43.43%
60/40,9.85%,16.67%,0.562,-43.88%
Tech,11.79%,16.10%,0.685,-38.46%


## Maximum leverage

The maximum allowed portfolio leverage is varied between **1x, 1.5x and 2x**.

The volatility target remains fixed at 10%.

In [7]:
leverage_caps = {
    "1x": 1.0,
    "1.5x": 1.5,
    "2x": 2.0
}

leverage_returns = {}

for label, max_leverage in leverage_caps.items():
    leverage_returns[label] = run_robustness_returns(max_leverage=max_leverage)

common_index = (
    leverage_returns["1x"].index
    .intersection(leverage_returns["1.5x"].index)
    .intersection(leverage_returns["2x"].index)
)

for label, port_returns in leverage_returns.items():
    aligned = port_returns.loc[common_index]
    result_table = performance_table(aligned)

    display(
        result_table.style
        .format({
            "Annual Return": "{:.2%}",
            "Annual Volatility": "{:.2%}",
            "Sharpe Ratio": "{:.3f}",
            "Drawdown": "{:.2%}"
        })
        .set_caption(f"Maximum Leverage — {label}")
    )

,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.28%,10.38%,0.411,-30.50%
Inverse Vol,3.77%,4.94%,0.486,-15.47%
Max Geo,8.03%,10.57%,0.650,-24.03%
Max Sharpe,5.83%,6.99%,0.643,-17.17%
Carver,3.17%,3.38%,0.521,-10.07%
Hierarchical Inv Vol,3.33%,3.69%,0.521,-10.25%
SPY,6.63%,11.06%,0.507,-30.91%
60/40,7.09%,9.60%,0.613,-27.74%
Tech,8.47%,10.64%,0.683,-26.99%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.23%,10.86%,0.393,-30.50%
Inverse Vol,4.32%,7.42%,0.415,-22.95%
Max Geo,8.03%,10.57%,0.650,-24.03%
Max Sharpe,6.34%,7.74%,0.650,-19.92%
Carver,3.50%,5.07%,0.423,-15.81%
Hierarchical Inv Vol,3.72%,5.54%,0.431,-16.08%
SPY,6.63%,11.06%,0.507,-30.91%
60/40,7.42%,11.10%,0.572,-31.05%
Tech,8.47%,10.64%,0.683,-26.99%


,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
Strategy,,,,
Equal Weight,5.23%,10.86%,0.393,-30.50%
Inverse Vol,4.06%,9.01%,0.329,-29.73%
Max Geo,8.03%,10.57%,0.650,-24.03%
Max Sharpe,6.72%,8.40%,0.648,-22.58%
Carver,3.76%,6.73%,0.371,-21.25%
Hierarchical Inv Vol,4.00%,7.30%,0.379,-21.59%
SPY,6.63%,11.06%,0.507,-30.91%
60/40,7.41%,11.11%,0.571,-31.05%
Tech,8.47%,10.64%,0.683,-26.99%
